In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
import ollama
import numpy as np
import dspy

local_llm = dspy.LM(
    "openai/qwen3:30b", 
    api_base="http://localhost:11434/v1", 
    api_key="no_key_needed"
)

dspy.configure(lm=local_llm,  cache=False)
dspy.configure_cache(enable_disk_cache=False)
dspy.configure_cache(enable_memory_cache=False)

In [2]:
# Beispieldaten definieren
documents = [
    "Titus Skates wurde in den späten 1970er Jahren von dem deutschen Skateboard-Pionier Titus Dittmann gegründet. Er trug maßgeblich zur Popularisierung der Skateboard-Kultur in Deutschland bei und etablierte mehrere Skate-Shops.",
    "Der Münsteraner Unternehmer Titus Dittmann war nicht nur Gründer der Firma Titus, sondern organisierte später auch den legendären Münster Monster Mastership, einen der wichtigsten Skateboard-Contests Europas.",
    "Skateboards bestehen typischerweise aus sieben Lagen kanadischem Ahornholz. Die Rollen sind meist aus Polyurethan, und die Achsen werden aus Aluminium gefertigt.",
    "In den 1980er Jahren entstanden in Deutschland zahlreiche Skate-Shops, darunter bekannte Marken wie Titus, City Skates und weitere unabhängige Stores. Allerdings wurden nicht alle von denselben Personen gegründet.",
    "Titus Dittmann setzte sich später vermehrt für soziale Projekte ein, unter anderem für Skate-Aid, ein weltweites Kinderhilfsprojekt, das Skateboarding als pädagogisches Werkzeug einsetzt.",
	"Apple wurde 1976 von Steve Jobs, Steve Wozniak und Ronald Wayne gegründet. Das Unternehmen begann in einer Garage in Los Altos.",
	"Die Stadt Münster ist bekannt für ihre historische Altstadt, das Picasso-Museum und eine der größten Universitäten Deutschlands.",
	"Die Firma Vans, eine der einflussreichsten Marken im Skateboarding, wurde 1966 von Paul Van Doren gegründet.",
    "Die Skateboard-Kultur in Deutschland wurde stark von amerikanischen Profi-Skatern beeinflusst, die in den 1980ern auf Europa-Tour waren.",
    "Der Mond umkreist die Erde in etwa 27,3 Tagen. Seine Gravitation verursacht die Gezeiten auf unserem Planeten."
]

In [3]:
# Qdrant initialisieren (In-Memory für Tests, persistent via Pfad möglich)
qdrant_client = QdrantClient(":memory:")
collection_name = "my_knowledge_base"

# Collection erstellen
qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

# Daten vektorisieren und hochladen
points = []
for idx, doc in enumerate(documents):
    vector = ollama.embed(model="embeddinggemma", input=doc).embeddings
    points.append(PointStruct(id=idx, vector=vector, payload={"text": doc}))

qdrant_client.upsert(
    collection_name=collection_name,
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [4]:
from typing import List, Union

#Implementierung eines Qdrant-Retrievers für DSPy
class QdrantRM(dspy.Retrieve):
    def __init__(self, client, collection_name, encoder, k=3):
        super().__init__(k=k)
        self.client = client
        self.collection_name = collection_name
        self.encoder = encoder

    def forward(self, query_or_queries: Union[str, List[str]], k=None) -> dspy.Prediction:
        k = k if k is not None else self.k

        # Falls mehrere Queries kommen, nehmen wir hier vereinfacht die erste oder verarbeiten eine Liste
        # DSPy übergibt oft einen einzelnen String
        query = query_or_queries if isinstance(query_or_queries, str) else query_or_queries[0]

        # Query vektorisieren
        query_vector = ollama.embed(model="embeddinggemma", input=doc).embeddings[0]

        # Suche in Qdrant
        search_result = qdrant_client.query_points(
            collection_name=self.collection_name,
            query=np.array(query_vector),
            limit=k
        )
        
        # Extrahieren der Text-Passagen aus dem Payload
        passages = [hit.payload['text'] for hit in search_result.points]

        # Rückgabe als DSPy Prediction
        return dspy.Prediction(passages=passages)

In [5]:
# Signatur
class GenerateAnswer(dspy.Signature):
    """Beantworte Fragen basierend auf dem gegebenen Kontext."""

    context = dspy.InputField(desc="Fakten aus der Wissensdatenbank")
    question = dspy.InputField(desc="Die Frage des Nutzers")
    answer = dspy.OutputField(desc="Die präzise Antwort")

In [6]:
# RAG Modul
class RAG(dspy.Module):
    def __init__(self, retriever_model):
        super().__init__()
        self.retrieve = retriever_model
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        # 1. Informationen abrufen (Retrieval)
        retrieval_result = self.retrieve(question)
        context = retrieval_result.passages

        # 2. Antwort generieren (Generation)
        prediction = self.generate(context=context, question=question)

        return dspy.Prediction(context=context, answer=prediction.answer)

In [7]:
# Initialisieren des Retrievers
my_retriever = QdrantRM(
    client=qdrant_client, 
    collection_name=collection_name, 
    encoder=ollama, 
    k=2
)

# Initialisieren des RAG-Systems
rag_system = RAG(retriever_model=my_retriever)

# Testfrage stellen
question = "Wer gründete die Firma Titus Skates?"

# Ausführung
response = rag_system(question)

# Ausgabe der Ergebnisse
print(f"Frage: {question}")
print(f"Gefundener Kontext: {response.context}")
print(f"Antwort: {response.answer}")

Frage: Wer gründete die Firma Titus Skates?
Gefundener Kontext: ['Der Mond umkreist die Erde in etwa 27,3 Tagen. Seine Gravitation verursacht die Gezeiten auf unserem Planeten.', 'Der Münsteraner Unternehmer Titus Dittmann war nicht nur Gründer der Firma Titus, sondern organisierte später auch den legendären Münster Monster Mastership, einen der wichtigsten Skateboard-Contests Europas.']
Antwort: Titus Dittmann


In [8]:
# Optional: Inspektion der Gedankengänge
local_llm.inspect_history(n=10)





[2025-11-29T11:00:30.700460]

System message:

Your input fields are:
1. `context` (str): Fakten aus der Wissensdatenbank
2. `question` (str): Die Frage des Nutzers
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): Die präzise Antwort
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Beantworte Fragen basierend auf dem gegebenen Kontext.


User message:

[[ ## context ## ]]
[1] «Der Mond umkreist die Erde in etwa 27,3 Tagen. Seine Gravitation verursacht die Gezeiten auf unserem Planeten.»
[2] «Der Münsteraner Unternehmer Titus Dittmann war nicht nur Gründer der Firma Titus, sondern organisierte später auch den legendären Münster Monster Mastership, einen der wichtigsten Skateboard-Contests Europas.»

[[ ## questio